## 1. Import knižníc

# Trénovanie CNN pre predikciu slnečného žiarenia

Tento notebook trénuje konvolučné neurónové siete na predikciu slnečného žiarenia z RGB obrázkov pomocou Intel Arc GPU akcelerácie.

In [1]:
import sys
from pathlib import Path

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import torch

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from models.cnn import create_cnn_model
from models.dataset import create_data_loaders
from models.utils import get_device, count_parameters
from models.training import (
    train_model,
    evaluate_model,
    create_optimizer_and_criterion,
    load_checkpoint,
)
from utils.evaluation import (
    plot_training_history,
    plot_residuals,
    plot_predictions_comparison,
)
print(f"Verzia PyTorch: {torch.__version__}")

try:
    import intel_extension_for_pytorch as ipex
    print(f"Intel Extension for PyTorch: {ipex.__version__}")
except (ImportError, OSError) as e:
    print(f"Intel Extension for PyTorch nie je dostupné: {e}")

if hasattr(torch, 'xpu') and torch.xpu.is_available():
    print(f"Intel XPU dostupné: {torch.xpu.device_count()} zariadení")
    for i in range(torch.xpu.device_count()):
        print(f"  [{i}]: {torch.xpu.get_device_name(i)}")
else:
    print("Používa sa CPU (GPU nebolo detekované)")


device = get_device()


Verzia PyTorch: 2.7.0+xpu
Intel Extension for PyTorch: 2.7.10+xpu
Intel XPU dostupné: 1 zariadení
  [0]: Intel(R) Arc(TM) B580 Graphics
Intel Arc GPU detected: Intel(R) Arc(TM) B580 Graphics


## 2. Načítanie spracovaných dát

In [2]:
data_dir = project_root / "dataset" / "Solar_data"
processed_dir = project_root / "dataset" / "processed"

train_df = pl.read_csv(processed_dir / "train.csv")
val_df = pl.read_csv(processed_dir / "val.csv")
test_df = pl.read_csv(processed_dir / "test.csv")

print(f"Trénovacie vzorky: {len(train_df)}")
print(f"Validačné vzorky: {len(val_df)}")
print(f"Testovacie vzorky: {len(test_df)}")


Trénovacie vzorky: 2159
Validačné vzorky: 462
Testovacie vzorky: 464


## 3. Definícia hyperparametrických konfigurácií

Budeme experimentovať s aspoň 5 rôznymi konfiguráciami s variáciami:
- Počet kanálov konvolučných vrstiev
- Veľkosti plne prepojených vrstiev
- Dropout hodnoty
- Learning rate hodnoty
- Batch size hodnoty

In [3]:
configs = [
    {
        "name": "Konfigurácia 1: Základná",
        "conv_channels": [32, 64, 128],
        "fc_hidden": [512, 256],
        "dropout": 0.5,
        "learning_rate": 0.001,
        "batch_size": 96,
        "epochs": 10,
    },
    {
        "name": "Konfigurácia 2: Hlbšia sieť",
        "conv_channels": [32, 64, 128, 256],
        "fc_hidden": [1024, 512, 256],
        "dropout": 0.5,
        "learning_rate": 0.001,
        "batch_size": 96,
        "epochs": 10,
    },
    {
        "name": "Konfigurácia 3: Vyšší dropout",
        "conv_channels": [32, 64, 128],
        "fc_hidden": [512, 256],
        "dropout": 0.7,
        "learning_rate": 0.001,
        "batch_size": 96,
        "epochs": 10,
    },
    {
        "name": "Konfigurácia 4: Nižší learning rate",
        "conv_channels": [32, 64, 128],
        "fc_hidden": [512, 256],
        "dropout": 0.5,
        "learning_rate": 0.0001,
        "batch_size": 96,
        "epochs": 10,
    },
    {
        "name": "Konfigurácia 5: Väčší batch size",
        "conv_channels": [32, 64, 128],
        "fc_hidden": [512, 256],
        "dropout": 0.5,
        "learning_rate": 0.001,
        "batch_size": 128,
        "epochs": 10,
    },
]

print(f"Celkový počet konfigurácií na testovanie: {len(configs)}")

Celkový počet konfigurácií na testovanie: 5


## 4. Trénovanie všetkých konfigurácií

In [ ]:
results = []
checkpoint_dir = project_root / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

for idx, config in enumerate(configs, 1):
    print(f"\n{'='*80}")
    print(f"Trénovanie {config['name']} ({idx}/{len(configs)})")
    print(f"{'='*80}")
    print(f"Parametre: {config}")
    
    train_loader, val_loader, test_loader = create_data_loaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        data_dir=data_dir,
        batch_size=config['batch_size'],
        num_workers=0,
    )
    
    model = create_cnn_model(
        conv_channels=config['conv_channels'],
        fc_hidden=config['fc_hidden'],
        dropout_rate=config['dropout'],
        device=device,
    )
    
    print(f"\nPočet parametrov modelu: {count_parameters(model):,}")
    
    optimizer, criterion = create_optimizer_and_criterion(
        model,
        learning_rate=config['learning_rate'],
        weight_decay=1e-4,
    )
    
    config_checkpoint_dir = checkpoint_dir / f"config_{idx}"
    config_checkpoint_dir.mkdir(exist_ok=True)
    
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        epochs=config['epochs'],
        device=device,
        checkpoint_dir=config_checkpoint_dir,
        early_stopping_patience=5,
        verbose=True,
    )
    
    best_model_path = config_checkpoint_dir / "best_model.pth"
    if best_model_path.exists():
        load_checkpoint(model, optimizer, best_model_path, device)
        print(f"\nNačítaný najlepší model z {best_model_path}")
    
    print("\nVyhodnocovanie na trénovacej množine...")
    train_metrics = evaluate_model(model, train_loader, criterion, device)
    
    print("Vyhodnocovanie na testovacej množine...")
    test_metrics = evaluate_model(model, test_loader, criterion, device)
    
    result = {
        "config_name": config['name'],
        "config_idx": idx,
        "config": config,
        "history": history,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "num_parameters": count_parameters(model),
    }
    results.append(result)
    
    print(f"\nFinálne výsledky pre {config['name']}:")
    print(f"  Trénovacia - MSE: {train_metrics['mse']:.4f}, MAE: {train_metrics['mae']:.4f}, "
          f"RMSE: {train_metrics['rmse']:.4f}, R²: {train_metrics['r2']:.4f}")
    print(f"  Testovacia  - MSE: {test_metrics['mse']:.4f}, MAE: {test_metrics['mae']:.4f}, "
          f"RMSE: {test_metrics['rmse']:.4f}, R²: {test_metrics['r2']:.4f}")

print(f"\n{'='*80}")
print("Všetky konfigurácie úspešne natrénované!")
print(f"{'='*80}")



Trénovanie Konfigurácia 1: Základná (1/5)
Parametre: {'name': 'Konfigurácia 1: Základná', 'conv_channels': [32, 64, 128], 'fc_hidden': [512, 256], 'dropout': 0.5, 'learning_rate': 0.001, 'batch_size': 96, 'epochs': 10}

Počet parametrov modelu: 3,631,265
Model and optimizer optimized with Intel Extension for PyTorch (O1)


Epoch 1: 100%|██████████| 23/23 [00:50<00:00,  2.22s/it, loss=2.31e+4]



Epoch 1/10:
  Train - Loss: 37561.2647, MSE: 37561.2617, MAE: 117.4649, RMSE: 193.8073, R2: 0.3682
  Val   - Loss: 66215.1814, MSE: 66215.1797, MAE: 175.0015, RMSE: 257.3231, R2: -0.0300


Epoch 2:  70%|██████▉   | 16/23 [00:30<00:13,  1.96s/it, loss=1.76e+4]

## 5. Porovnávacia tabuľka výsledkov

In [ ]:
table_data = []
for result in results:
    table_data.append(
        {
            "Konfigurácia": result["config_name"],
            "Parametre": f"{result['num_parameters']:,}",
            "Train MSE": f"{result['train_metrics']['mse']:.4f}",
            "Train MAE": f"{result['train_metrics']['mae']:.4f}",
            "Train RMSE": f"{result['train_metrics']['rmse']:.4f}",
            "Train R²": f"{result['train_metrics']['r2']:.4f}",
            "Test MSE": f"{result['test_metrics']['mse']:.4f}",
            "Test MAE": f"{result['test_metrics']['mae']:.4f}",
            "Test RMSE": f"{result['test_metrics']['rmse']:.4f}",
            "Test R²": f"{result['test_metrics']['r2']:.4f}",
        }
    )

results_df = pl.DataFrame(table_data)

print("\nPorovnanie výsledkov:")

print(results_df)

results_df.write_csv(project_root / "results_table.csv")
print(f"\nVýsledky uložené do {project_root / 'results_table.csv'}")


## 6. Výber najlepšej konfigurácie

In [ ]:
best_idx = np.argmax([r['test_metrics']['r2'] for r in results])
best_result = results[best_idx]

print(f"Najlepšia konfigurácia: {best_result['config_name']}")
print(f"Test R²: {best_result['test_metrics']['r2']:.4f}")
print(f"Test RMSE: {best_result['test_metrics']['rmse']:.4f}")
print(f"Test MAE: {best_result['test_metrics']['mae']:.4f}")
print(f"Test MSE: {best_result['test_metrics']['mse']:.4f}")

## 7. História trénovania najlepšieho modelu

In [ ]:
fig_dir = project_root / "figures"
fig_dir.mkdir(exist_ok=True)

fig = plot_training_history(
    best_result['history'],
    save_path=fig_dir / f"training_history_config_{best_result['config_idx']}.png"
)
plt.show()
print(f"História trénovania uložená do {fig_dir}")

## 8. Analýza reziduí najlepšieho modelu

In [ ]:
fig = plot_residuals(
    targets=best_result['test_metrics']['targets'],
    predictions=best_result['test_metrics']['predictions'],
    split_name="Test",
    save_path=fig_dir / f"residuals_config_{best_result['config_idx']}.png"
)
plt.show()
print(f"Grafy reziduí uložené do {fig_dir}")

## 9. Porovnanie predikcií najlepšieho modelu

In [ ]:
fig = plot_predictions_comparison(
    targets=best_result['test_metrics']['targets'],
    predictions=best_result['test_metrics']['predictions'],
    split_name="Test",
    n_samples=100,
    save_path=fig_dir / f"predictions_comparison_config_{best_result['config_idx']}.png"
)
plt.show()
print(f"Porovnanie predikcií uložené do {fig_dir}")

## 10. História trénovania všetkých konfigurácií

In [ ]:
for result in results:
    print(f"\nVykresľovanie histórie trénovania pre {result['config_name']}")
    fig = plot_training_history(
        result['history'],
        save_path=fig_dir / f"training_history_config_{result['config_idx']}.png"
    )
    plt.show()
    plt.close()

print(f"\nVšetky histórie trénovania uložené do {fig_dir}")

## 11. Súhrnné štatistiky

In [ ]:
print("\n" + "="*80)
print("FINÁLNE ZHRNUTIE")
print("="*80)

print(f"\nNajlepšia konfigurácia: {best_result['config_name']}")
print(f"Počet parametrov: {best_result['num_parameters']:,}")
print(f"\nDetail konfigurácie:")
for key, value in best_result['config'].items():
    print(f"  {key}: {value}")

print(f"\nVýkon najlepšieho modelu:")
print(f"  Trénovacia množina:")
print(f"    MSE:  {best_result['train_metrics']['mse']:.4f}")
print(f"    MAE:  {best_result['train_metrics']['mae']:.4f}")
print(f"    RMSE: {best_result['train_metrics']['rmse']:.4f}")
print(f"    R²:   {best_result['train_metrics']['r2']:.4f}")
print(f"\n  Testovacia množina:")
print(f"    MSE:  {best_result['test_metrics']['mse']:.4f}")
print(f"    MAE:  {best_result['test_metrics']['mae']:.4f}")
print(f"    RMSE: {best_result['test_metrics']['rmse']:.4f}")
print(f"    R²:   {best_result['test_metrics']['r2']:.4f}")

print(f"\nVšetky výsledky a obrázky uložené do:")
print(f"  - Tabuľka výsledkov: {project_root / 'results_table.csv'}")
print(f"  - Obrázky: {fig_dir}")
print(f"  - Kontrolné body: {checkpoint_dir}")
print("="*80)


## 12. Bonus: Vizualizácia konvolučných filtrov

In [ ]:
from utils.bonus import visualize_conv_filters

best_model = create_cnn_model(
    conv_channels=best_result['config']['conv_channels'],
    fc_hidden=best_result['config']['fc_hidden'],
    dropout_rate=best_result['config']['dropout'],
    device=device,
)

best_model_path = checkpoint_dir / f"config_{best_result['config_idx']}" / "best_model.pth"
optimizer_dummy, _ = create_optimizer_and_criterion(best_model, learning_rate=0.001)
load_checkpoint(best_model, optimizer_dummy, best_model_path, device)
best_model.eval()

print(f"Načítaný najlepší model: {best_result['config_name']}")
print(f"Počet konvolučných blokov: {len(best_model.conv_layers)}")

print("\n" + "="*60)
print("Vizualizácia filtrov prvej konvolučnej vrstvy")
print("="*60)
save_path = fig_dir / f"conv_filters_layer_0_config_{best_result['config_idx']}.png"
visualize_conv_filters(best_model, layer_idx=0, max_filters=32, save_path=save_path)
plt.show()

if len(best_model.conv_layers) > 1:
    print("\n" + "="*60)
    print("Vizualizácia filtrov druhej konvolučnej vrstvy")
    print("="*60)
    save_path = fig_dir / f"conv_filters_layer_1_config_{best_result['config_idx']}.png"
    visualize_conv_filters(best_model, layer_idx=1, max_filters=32, save_path=save_path)
    plt.show()


## 13. Bonus: Testovanie na vlastných obrázkoch oblohy

In [ ]:
from utils.bonus import predict_custom_images

my_images_dir = project_root / "dataset" / "Solar_data" / "my_images"
my_images_dir.mkdir(parents=True, exist_ok=True)

print("="*80)
print("TESTOVANIE NA VLASTNÝCH OBRÁZKOCH OBLOHY")
print("="*80)
print(f"\nPriečinok s obrázkami: {my_images_dir}")

custom_results = predict_custom_images(best_model, my_images_dir, device)

if len(custom_results) >= 5:
    print(f"\nÚspešne spracovaných {len(custom_results)} vlastných obrázkov")
    print("Bonusová úloha splnená!")
elif len(custom_results) > 0:
    print(f"\nNájdených len {len(custom_results)} obrázkov (minimum je 5)")
else:
    print("\nŽiadne obrázky neboli nájdené. Prosím pridajte vlastné fotografie oblohy do:")
    print(f"  {my_images_dir}")


### 14.1 Vizualizácia predikcií vlastných obrázkov

In [ ]:
from utils.bonus import visualize_custom_predictions

if len(custom_results) > 0:
    save_path = fig_dir / f"custom_images_predictions_config_{best_result['config_idx']}.png"
    visualize_custom_predictions(custom_results, save_path=save_path)
    plt.show()
else:
    print("Žiadne vlastné obrázky na vizualizáciu. Pridajte obrázky do priečinka my_images a spustite túto bunku znovu.")
